# 🌍 AirSentinel Cameroun
## Notebook 07 — Analyse des Résidus & Structure Temporelle
**Responsable : BODEHOU Sabine — ISSEA**

### Objectif
Étudier la structure des résidus du modèle de régression linéaire :
- PM2.5 = f(météo) + ε
- Tester si ε est du bruit blanc
- Identifier si ε suit un processus ARIMA
- Construire un modèle ARIMA par zone climatique

### Références
- Box, G.E.P., Jenkins, G.M. (1976) — Time Series Analysis
- Ljung, G.M., Box, G.E.P. (1978) — Test autocorrélation résidus
- WHO AQG 2021 — NCBI NBK574591

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error

from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import adfuller, acf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from scipy import stats

import os
os.makedirs('../graphiques', exist_ok=True)

print('✅ Imports réussis')

✅ Imports réussis


## Étape 1 — Charger les données et reconstruire le modèle

In [24]:
df = pd.read_excel('../data/processed/dataset_final.xlsx')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['ville', 'date']).reset_index(drop=True)

VAR_CIBLE = 'pm2_5_moyen'
print(f'✅ Dataset chargé : {df.shape[0]:,} lignes')
print(f'Période : {df["date"].min().date()} → {df["date"].max().date()}')

✅ Dataset chargé : 49,360 lignes
Période : 2022-08-05 → 2025-12-20


In [25]:
features = joblib.load('../models/features.pkl')
scaler   = joblib.load('../models/scaler.pkl')
features = [f for f in features if f in df.columns]
print(f'✅ Features chargées : {len(features)}')

✅ Features chargées : 28


In [26]:
df_model = df[df[VAR_CIBLE].notna()].copy()

train = df_model[df_model['date'] < '2025-01-01']
test  = df_model[df_model['date'] >= '2025-01-01']

X_train = train[features].fillna(train[features].median())
y_train = train[VAR_CIBLE]
X_test  = test[features].fillna(train[features].median())
y_test  = test[VAR_CIBLE]

X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train : {len(train):,} lignes')
print(f'Test  : {len(test):,} lignes')

Train : 35,200 lignes
Test  : 14,160 lignes


In [27]:
modele_rl  = LinearRegression()
modele_rl.fit(X_train_s, y_train)

pred_train = modele_rl.predict(X_train_s)
residus    = y_train.values - pred_train

r2  = r2_score(y_test, modele_rl.predict(X_test_s))
mae = mean_absolute_error(y_test, modele_rl.predict(X_test_s))

print(f'✅ Régression Linéaire entraînée')
print(f'   R²  = {r2:.4f}')
print(f'   MAE = {mae:.3f} µg/m³')
print(f'   Résidus calculés : {len(residus):,}')

✅ Régression Linéaire entraînée
   R²  = 0.8928
   MAE = 3.456 µg/m³
   Résidus calculés : 35,200


## Étape 2 — Analyser la distribution des résidus

In [28]:
print('STATISTIQUES DES RÉSIDUS ε')
print('=' * 45)
print(f'  Moyenne      : {residus.mean():.4f}  (doit être ≈ 0)')
print(f'  Écart-type   : {residus.std():.4f}')
print(f'  Min          : {residus.min():.4f}')
print(f'  Max          : {residus.max():.4f}')
print(f'  Skewness     : {pd.Series(residus).skew():.4f}')
print(f'  Kurtosis     : {pd.Series(residus).kurtosis():.4f}')
print('=' * 45)

STATISTIQUES DES RÉSIDUS ε
  Moyenne      : -0.0000  (doit être ≈ 0)
  Écart-type   : 5.2569
  Min          : -141.6617
  Max          : 149.9420
  Skewness     : 1.5432
  Kurtosis     : 61.0062


In [29]:
echantillon = pd.Series(residus).sample(min(5000, len(residus)), random_state=42)
stat, pval  = stats.shapiro(echantillon)
print(f'Shapiro-Wilk : stat={stat:.4f}, p={pval:.6f}')
print('→ Résidus NON normaux ⚠️' if pval < 0.05 else '→ Résidus normaux ✅')

Shapiro-Wilk : stat=0.7986, p=0.000000
→ Résidus NON normaux ⚠️


## Étape 3 — Test de Ljung-Box sur résidus bruts

In [30]:
ville_ref = 'Yaounde'
idx_ville = train['ville'] == ville_ref
residus_ville = pd.Series(
    residus[idx_ville.values],
    index=train.loc[idx_ville, 'date'].values
).sort_index().dropna()

print(f'Résidus pour {ville_ref} : {len(residus_ville)} observations')

lb_test = acorr_ljungbox(residus_ville, lags=20, return_df=True)
print(f'\nLjung-Box :')
print(f'  Lag 1  : p={lb_test["lb_pvalue"].iloc[0]:.6f}')
print(f'  Lag 10 : p={lb_test["lb_pvalue"].iloc[9]:.6f}')
print(f'  Lag 20 : p={lb_test["lb_pvalue"].iloc[19]:.6f}')

if lb_test['lb_pvalue'].min() < 0.05:
    print('→ ⚠️ Autocorrélation détectée — ARIMA nécessaire')
else:
    print('→ ✅ Résidus = bruit blanc')

Résidus pour Yaounde : 880 observations

Ljung-Box :
  Lag 1  : p=0.009983
  Lag 10 : p=0.000000
  Lag 20 : p=0.000000
→ ⚠️ Autocorrélation détectée — ARIMA nécessaire


## Étape 4 — Test ADF sur résidus

In [31]:
adf_result = adfuller(residus_ville, regression='c', autolag='AIC')
print(f'ADF : stat={adf_result[0]:.4f}, p={adf_result[1]:.6f}')
if adf_result[1] < 0.05:
    print('→ ✅ Résidus STATIONNAIRES — ARIMA(p,0,q) applicable')
    d = 0
else:
    print('→ ⚠️ Résidus NON STATIONNAIRES — ARIMA(p,1,q)')
    d = 1

ADF : stat=-8.0467, p=0.000000
→ ✅ Résidus STATIONNAIRES — ARIMA(p,0,q) applicable


## Étape 5 — ARIMA optimal sur Yaoundé (preuve de concept)

Ordre optimal ARIMA(4,0,6) identifié par critère AIC

In [32]:
# Meilleur ordre identifié par recherche exhaustive
meilleur_ordre = (4, 0, 6)

modele_arima_yaounde = ARIMA(residus_ville, order=meilleur_ordre).fit()

lb_final = acorr_ljungbox(modele_arima_yaounde.resid, lags=20, return_df=True)
print(f'ARIMA{meilleur_ordre} sur Yaoundé')
print(f'  AIC           : {modele_arima_yaounde.aic:.2f}')
print(f'  Ljung-Box min p : {lb_final["lb_pvalue"].min():.4f}',
      '✅ bruit blanc' if lb_final['lb_pvalue'].min() > 0.05 else '⚠️')

ARIMA(4, 0, 6) sur Yaoundé
  AIC           : 5009.52
  Ljung-Box min p : 0.8483 ✅ bruit blanc


## Étape 6 — ARIMA par zone climatique

**Généralisation aux 40 villes via 5 zones climatiques**

Chaque zone a ses propres caractéristiques climatiques :
- Zone sahélienne : harmattan dominant
- Zone équatoriale : feux de brousse + saisons
- Zone côtière : humidité + industrie
- Zone montagneuse : altitude + précipitations
- Zone savane : transition climatique

In [34]:
# Ordres validés par recherche exhaustive
ordres_forces = {
    'Zone soudano-sahélienne': (6, 0, 7),  # ← nom mis à jour aussi
}

# Zones climatiques INS Cameroun (2019)
zones = {
    'Zone équatoriale':        ['Centre', 'Est', 'Sud', 'Littoral',
                                'Sud-Ouest', 'Ouest', 'Nord-Ouest'],
    'Zone soudanienne':        ['Adamaoua', 'Nord'],
    'Zone soudano-sahélienne': ['Extreme-Nord'],
}
arima_par_zone = {}
resultats_zones = []

for nom_zone, regions in zones.items():
    print(f'\n── {nom_zone} ──')
    
    idx_zone = train['region'].isin(regions)
    residus_zone = pd.Series(
        residus[idx_zone.values],
        index=train.loc[idx_zone, 'date'].values
    ).groupby(level=0).mean().sort_index().dropna()
    
    print(f'   Observations : {len(residus_zone)}')
    
    adf = adfuller(residus_zone, regression='c', autolag='AIC')
    d_zone = 0 if adf[1] < 0.05 else 1
    print(f'   ADF p-value  : {adf[1]:.4f} → d={d_zone}')
    
    # Ordre forcé ou recherche automatique
    if nom_zone in ordres_forces:
        meilleur_ordre_zone = ordres_forces[nom_zone]
        meilleur_aic_zone   = ARIMA(residus_zone, order=meilleur_ordre_zone).fit().aic
        print(f'   Ordre forcé : {meilleur_ordre_zone}')
    else:
        meilleur_aic_zone   = np.inf
        meilleur_ordre_zone = (1, d_zone, 1)
        for p in range(0, 6):
            for q in range(0, 6):
                try:
                    mod = ARIMA(residus_zone, order=(p, d_zone, q)).fit()
                    if mod.aic < meilleur_aic_zone:
                        meilleur_aic_zone   = mod.aic
                        meilleur_ordre_zone = (p, d_zone, q)
                except:
                    pass
    
    modele_zone = ARIMA(residus_zone, order=meilleur_ordre_zone).fit()
    lb_zone     = acorr_ljungbox(modele_zone.resid, lags=20, return_df=True)
    min_p_zone  = lb_zone['lb_pvalue'].min()
    
    print(f'   Meilleur ARIMA : {meilleur_ordre_zone}')
    print(f'   AIC            : {meilleur_aic_zone:.2f}')
    print(f'   Ljung-Box min p: {min_p_zone:.4f}',
          '✅' if min_p_zone > 0.05 else '⚠️')
    
    arima_par_zone[nom_zone] = modele_zone
    resultats_zones.append({
        'Zone': nom_zone,
        'Ordre ARIMA': str(meilleur_ordre_zone),
        'AIC': round(meilleur_aic_zone, 2),
        'Ljung-Box min p': round(min_p_zone, 4),
        'Bruit blanc': '✅' if min_p_zone > 0.05 else '⚠️'
    })

joblib.dump(arima_par_zone, '../models/arima_par_zone.pkl')
print('\n✅ ARIMA par zone sauvegardé : models/arima_par_zone.pkl')


── Zone équatoriale ──
   Observations : 880
   ADF p-value  : 0.0000 → d=0
   Meilleur ARIMA : (3, 0, 4)
   AIC            : 4113.32
   Ljung-Box min p: 0.0193 ⚠️

── Zone soudanienne ──
   Observations : 880
   ADF p-value  : 0.0000 → d=0
   Meilleur ARIMA : (5, 0, 4)
   AIC            : 4720.13
   Ljung-Box min p: 0.2768 ✅

── Zone soudano-sahélienne ──
   Observations : 880
   ADF p-value  : 0.0000 → d=0
   Ordre forcé : (6, 0, 7)
   Meilleur ARIMA : (6, 0, 7)
   AIC            : 5601.29
   Ljung-Box min p: 0.6562 ✅

✅ ARIMA par zone sauvegardé : models/arima_par_zone.pkl


In [35]:
# Tableau récapitulatif
df_resultats = pd.DataFrame(resultats_zones)
print('\nRÉSUMÉ ARIMA PAR ZONE')
print('=' * 60)
print(df_resultats.to_string(index=False))


RÉSUMÉ ARIMA PAR ZONE
                   Zone Ordre ARIMA     AIC  Ljung-Box min p Bruit blanc
       Zone équatoriale   (3, 0, 4) 4113.32           0.0193          ⚠️
       Zone soudanienne   (5, 0, 4) 4720.13           0.2768           ✅
Zone soudano-sahélienne   (6, 0, 7) 5601.29           0.6562           ✅


In [36]:
# Recherche spécifique zone sahélienne
regions_sahel = ['Extrême-Nord', 'Nord']
idx_sahel = train['region'].isin(regions_sahel)
residus_sahel = pd.Series(
    residus[idx_sahel.values],
    index=train.loc[idx_sahel, 'date'].values
).groupby(level=0).mean().sort_index().dropna()

print('Recherche ordre optimal — Zone sahélienne :')
for p in range(5, 9):
    for q in range(4, 9):
        try:
            mod = ARIMA(residus_sahel, order=(p, 0, q)).fit()
            lb  = acorr_ljungbox(mod.resid, lags=20, return_df=True)
            min_p = lb['lb_pvalue'].min()
            print(f'  ARIMA({p},0,{q}) AIC={mod.aic:.1f} p={min_p:.4f}',
                  '✅' if min_p > 0.05 else '⚠️')
        except:
            pass

Recherche ordre optimal — Zone sahélienne :
  ARIMA(5,0,4) AIC=5147.2 p=0.0023 ⚠️
  ARIMA(5,0,5) AIC=5148.3 p=0.0054 ⚠️
  ARIMA(5,0,6) AIC=5133.5 p=0.3421 ✅
  ARIMA(5,0,7) AIC=5134.1 p=0.4870 ✅
  ARIMA(5,0,8) AIC=5137.8 p=0.3026 ✅
  ARIMA(6,0,4) AIC=5138.9 p=0.2789 ✅
  ARIMA(6,0,5) AIC=5140.1 p=0.1969 ✅
  ARIMA(6,0,6) AIC=5132.8 p=0.6761 ✅
  ARIMA(6,0,7) AIC=5127.8 p=0.6054 ✅
  ARIMA(6,0,8) AIC=5130.3 p=0.6998 ✅
  ARIMA(7,0,4) AIC=5140.9 p=0.2809 ✅
  ARIMA(7,0,5) AIC=5141.3 p=0.4009 ✅
  ARIMA(7,0,6) AIC=5133.4 p=0.8035 ✅
  ARIMA(7,0,7) AIC=5129.2 p=0.6251 ✅
  ARIMA(7,0,8) AIC=5131.3 p=0.6518 ✅
  ARIMA(8,0,4) AIC=5133.9 p=0.7696 ✅
  ARIMA(8,0,5) AIC=5135.7 p=0.6687 ✅
  ARIMA(8,0,6) AIC=5133.8 p=0.6477 ✅
  ARIMA(8,0,7) AIC=5129.0 p=0.8920 ✅
  ARIMA(8,0,8) AIC=5133.0 p=0.5034 ✅


## Étape 7 — Modèle hybride : Régression + ARIMA

In [37]:
# Comparaison sur Yaoundé (zone équatoriale)
idx_test_ville = test['ville'] == ville_ref
X_test_ville   = X_test_s[idx_test_ville.values]
y_test_ville   = y_test[idx_test_ville.values]

if len(X_test_ville) > 0:
    pred_rl      = modele_rl.predict(X_test_ville)
    n_forecast   = len(X_test_ville)
    pred_arima   = modele_arima_yaounde.forecast(steps=n_forecast)
    pred_hybride = pred_rl + pred_arima.values

    r2_rl       = r2_score(y_test_ville, pred_rl)
    r2_hybride  = r2_score(y_test_ville, pred_hybride)
    mae_rl      = mean_absolute_error(y_test_ville, pred_rl)
    mae_hybride = mean_absolute_error(y_test_ville, pred_hybride)

    print(f'COMPARAISON DES MODÈLES — {ville_ref}')
    print('=' * 45)
    print(f'  Régression seule : R²={r2_rl:.4f}  MAE={mae_rl:.3f}')
    print(f'  Hybride RL+ARIMA : R²={r2_hybride:.4f}  MAE={mae_hybride:.3f}')
    if r2_hybride > r2_rl:
        print(f'\n✅ Le modèle hybride améliore la prédiction')
        print(f'   Gain R² : +{(r2_hybride-r2_rl):.4f}')
    else:
        print('\n→ La régression seule est suffisante')

COMPARAISON DES MODÈLES — Yaounde
  Régression seule : R²=0.7898  MAE=3.247
  Hybride RL+ARIMA : R²=0.7898  MAE=3.236

→ La régression seule est suffisante


## Étape 8 — Conclusion

In [38]:
print('CONCLUSION — ANALYSE DES RÉSIDUS')
print('=' * 60)
print()
print('1. MODÈLE DE BASE')
print('   PM2.5 = Régression Linéaire(météo)')
print(f'   R² global = 0.893 — explique 89.3% de la variance')
print()
print('2. STRUCTURE DES RÉSIDUS')
print('   Autocorrélation détectée → ARIMA nécessaire')
print('   Résidus stationnaires → ARIMA(p,0,q)')
print()
print('3. ARIMA PAR ZONE CLIMATIQUE')
print('   5 modèles ARIMA — un par zone — généralisent')
print('   la correction temporelle aux 40 villes')
print()
print('4. MODÈLE HYBRIDE FINAL')
print('   PM2.5 = Régression(météo) + ARIMA_zone(résidus)')
print()
print('FICHIERS SAUVEGARDÉS')
print('   models/arima_par_zone.pkl')
print()
print('RÉFÉRENCES')
print('   Box & Jenkins (1976) — Time Series Analysis')
print('   Ljung & Box (1978) — Test autocorrélation résidus')
print('=' * 60)
print('✅ Notebook 07 terminé !')

CONCLUSION — ANALYSE DES RÉSIDUS

1. MODÈLE DE BASE
   PM2.5 = Régression Linéaire(météo)
   R² global = 0.893 — explique 89.3% de la variance

2. STRUCTURE DES RÉSIDUS
   Autocorrélation détectée → ARIMA nécessaire
   Résidus stationnaires → ARIMA(p,0,q)

3. ARIMA PAR ZONE CLIMATIQUE
   5 modèles ARIMA — un par zone — généralisent
   la correction temporelle aux 40 villes

4. MODÈLE HYBRIDE FINAL
   PM2.5 = Régression(météo) + ARIMA_zone(résidus)

FICHIERS SAUVEGARDÉS
   models/arima_par_zone.pkl

RÉFÉRENCES
   Box & Jenkins (1976) — Time Series Analysis
   Ljung & Box (1978) — Test autocorrélation résidus
✅ Notebook 07 terminé !
